# Image embedding with ChromaDB

This notebook walks through the **embedding pipeline** for a single PNG: load pixels, run them through a vision model, inspect the resulting vector, and **persist** that vector in ChromaDB. We use CLIP via SentenceTransformers (`clip-ViT-B-32`), which turns the whole image into one fixed-length vector (512 dimensions here). That vector is a compressed numerical summary the model uses for similarity; it is not a transcript of text in the image.

The example file is `images/landscape.png` (an industry-style infographic). Change `IMAGE_PATH` to use another PNG. First run downloads model weights and may take several minutes.

In [ ]:
from pathlib import Path
from PIL import Image
import chromadb
from sentence_transformers import SentenceTransformer
from IPython.display import display, Image as IPImage

IMAGE_PATH = Path("images/landscape.png")
CHROMA_DIR = Path("chroma_image_data")

In [ ]:
img = Image.open(IMAGE_PATH).convert("RGB")
print("Path:", IMAGE_PATH)
print("Size (W×H):", img.size, "Mode:", img.mode)
display(IPImage(filename=str(IMAGE_PATH)))

In [ ]:
model = SentenceTransformer("clip-ViT-B-32")
emb = model.encode(img, normalize_embeddings=True)
d = int(emb.shape[0])
norm = float((emb ** 2).sum() ** 0.5)
preview = [round(float(x), 4) for x in emb[:8]]
print("Embedding length:", d, "(CLIP ViT-B/32)")
print("L2 norm after normalize_embeddings=True:", round(norm, 4), "(~1.0 expected)")
print("First 8 components:", preview)
vec = emb.tolist()

In [ ]:
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection(
    name="image_clip",
    metadata={"hnsw:space": "cosine"},
)
collection.upsert(
    ids=["img_0"],
    embeddings=[vec],
    metadatas=[{"path": str(IMAGE_PATH.resolve())}],
    documents=["user_png"],
)
print("Chroma collection:", collection.name, "| vectors:", collection.count())

## After this notebook: how it is used in practice

- **Scale up**: Embed many images (or video frames), store vectors and metadata in Chroma or another vector database, and use **approximate nearest-neighbor search** to retrieve similar items. CLIP can embed **text** with the same model, so you can also search images with short phrases (separate from this demo’s focus on the embedding step).
- **Production**: Add **filters** on metadata (tenant, date, product id), **batch / GPU** encoding jobs, **versioning** of embedding models, and **monitoring** (latency, drift). Very large corpora often use **dedicated vector indexes** and sharding.
- **When vectors are not enough**: Dense infographics, screenshots, or documents usually need **OCR**, **layout-aware models**, or **vision-language models** if you must answer factual questions about text inside pixels.

The important idea here: **pixels → model → fixed-size vector → database** is the same pattern most multimodal and image-search systems build on.